In [ ]:
# ============================================================
# pipeline_debug.ipynb
# 调试版流水线 —— 核心改动：save_matched_cells_v2（细胞级三层筛选）
# 其余逻辑与 pipeline.py 完全一致
# ============================================================

# %%
# ============================================================
# Block 1 ：配置参数（每次调试只需改这里）
# ============================================================
import torch
import numpy as np

# ── 路径配置 ────────────────────────────────────────────────
MODEL_PATH            = "/path/to/your/model.pth"
MALIGNANT_CELLS_DIR   = "/path/to/reference/malignant"
BENIGN_CELLS_DIR      = "/path/to/reference/benign"

# 候选域：直接指定单个病人文件夹（调试时一次只跑一个）
CANDIDATE_FOLDER      = "/path/to/patient_folder"   # 里面有 malignant_images/ 子目录
PATIENT_STATUS        = "negative"                  # "positive" or "negative"

BASE_SAVE_DIR         = "/path/to/save_results"

# ── 模型配置 ────────────────────────────────────────────────
MODEL_TYPE            = "ResNeXt"
FEATURE_LAYER         = "penultimate"
MEAN                  = None   # 或 [0.485, 0.456, 0.406]
STD                   = None   # 或 [0.229, 0.224, 0.225]

# ── 聚类配置 ────────────────────────────────────────────────
REFERENCE_K           = 10
MAX_CANDIDATE_K       = 20
PCA_DIM               = 32
BATCH_SIZE            = 16
NUM_WORKERS           = 0
CELL_SIZE_WEIGHT      = 1.0

# ── 恶性簇索引 & 匹配规则 ────────────────────────────────────
# 根据你的实际参考域聚类结果修改
MALIGNANT_REF_CLUSTERS = [8, 7, 1]

# 默认规则：每个恶性簇为 target，其余恶性簇为 avoid
MATCHING_RULES = [
    {
        "target"          : 8,
        "avoid"           : [7, 1],
        "threshold_target": 0.5,
        "threshold_ratio" : 3,
    },
    {
        "target"          : 7,
        "avoid"           : [8, 1],
        "threshold_target": 0.5,
        "threshold_ratio" : 3,
    },
    {
        "target"          : 1,
        "avoid"           : [8, 7],
        "threshold_target": 0.5,
        "threshold_ratio" : 3,
    },
]

# ── 【核心调参区】细胞级筛选参数 ─────────────────────────────
# 层1：细胞与最佳恶性参考簇的相似度必须 >= 该值
CELL_THRESHOLD_TARGET = 0.5

# 层2：细胞与最佳恶性参考簇的相似度，必须 >= 所有良性参考簇相似度的 N 倍
#      （注意：良性簇 = 非 MALIGNANT_REF_CLUSTERS 的所有簇）
CELL_THRESHOLD_RATIO  = 3.0

# 随机种子
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("✅ Block 1 配置完成")
print(f"   恶性参考簇: {MALIGNANT_REF_CLUSTERS}")
print(f"   细胞筛选 - 层1 threshold_target : {CELL_THRESHOLD_TARGET}")
print(f"   细胞筛选 - 层2 threshold_ratio  : {CELL_THRESHOLD_RATIO}")


# %%
# ============================================================
# Block 2 ：初始化模型（只需运行一次）
# ============================================================
import os
from torch.utils.data import DataLoader
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
from tqdm import tqdm

from cross_cluster_matching.models.models import get_model_and_transform
from cross_cluster_matching.data.data_utils import (
    collect_reference_data, collect_image_paths,
    CellImageDataset, extract_patient_id
)
from cross_cluster_matching.models.feature_extractor import extract_features_with_cell_size
from cross_cluster_matching.clustering.K_optimizer import optimize_k_for_candidate_clustering
from cross_cluster_matching.clustering.calculate_consensus_score import calculate_consensus_score
from cross_cluster_matching.clustering.matching_rule_application import (
    identify_matching_clusters, check_matching_rules
)
from cross_cluster_matching.visualization.visualization import (
    count_non_background_pixels,
    visualize_all_clusters_without_images,
    visualize_all_clusters_without_images_1,
    improved_visualize_clusters_with_images,
    visualize_matching_clusters,
    analyze_cluster_composition,
    plot_similarity_heatmap,
    visualize_filtered_clusters
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model, transform, _ = get_model_and_transform(
    MODEL_TYPE, device, MODEL_PATH, mean=MEAN, std=STD
)
print(f"✅ Block 2 模型加载完成，使用设备: {device}")


# %%
# ============================================================
# Block 3 ：处理参考域（只需运行一次，结果复用）
# ============================================================
print("======= 处理参考细胞 =======")

ref_paths, ref_sources = collect_reference_data(MALIGNANT_CELLS_DIR, BENIGN_CELLS_DIR)

ref_sizes = np.log1p([
    count_non_background_pixels(p)
    for p in tqdm(ref_paths, desc="Ref Sizes")
])

ref_loader = DataLoader(
    CellImageDataset(ref_paths, transform=transform, source_labels=ref_sources),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS
)
ref_feats, ref_sources, ref_paths = extract_features_with_cell_size(
    model, ref_loader, device, ref_sizes, FEATURE_LAYER, CELL_SIZE_WEIGHT
)

# PCA & KMeans
pca = PCA(n_components=min(PCA_DIM, ref_feats.shape[0], ref_feats.shape[1]))
ref_norm = normalize(pca.fit_transform(ref_feats))

kmeans_ref = KMeans(n_clusters=REFERENCE_K, random_state=SEED).fit(ref_norm)
ref_results = {
    'features'       : ref_norm,
    'labels'         : kmeans_ref.labels_,
    'source_labels'  : ref_sources,
    'image_paths'    : ref_paths,
    'cluster_centers': kmeans_ref.cluster_centers_,
    'n_clusters'     : REFERENCE_K,
    'pca'            : pca,
}

# 参考域可视化
ref_vis_dir = os.path.join(BASE_SAVE_DIR, f'reference_visualization_pca{PCA_DIM}')
os.makedirs(ref_vis_dir, exist_ok=True)

reference_umap = visualize_all_clusters_without_images(
    ref_results['features'], ref_results['labels'],
    ref_results['source_labels'], "Reference", ref_vis_dir
)
reference_umap_1 = visualize_all_clusters_without_images_1(
    ref_results['features'], ref_results['labels'],
    ref_results['source_labels'], "Reference", ref_vis_dir
)
improved_visualize_clusters_with_images(
    ref_results['features'], ref_results['labels'],
    ref_results['source_labels'], ref_results['image_paths'],
    "Reference", ref_vis_dir, ref_sizes
)
analyze_cluster_composition(
    ref_results['labels'], ref_results['source_labels'],
    ref_results['image_paths'], "Reference", ref_vis_dir
)

print(f"✅ Block 3 参考域处理完成，共 {len(ref_paths)} 个细胞，{REFERENCE_K} 个簇")
print(f"   参考域可视化保存至: {ref_vis_dir}")


# %%
# ============================================================
# Block 4 ：定义增强版 save_matched_cells_v2
# 【核心改动】在原函数基础上增加两层细胞级筛选
# ============================================================
import shutil

def save_matched_cells_v2(
    indices,
    image_paths,
    labels,
    cell_sim_matrix,          # shape=(N_cells, n_ref_clusters)
    rule_type,
    save_dirs,
    target_ref_indices,       # 恶性参考簇索引列表，如 [8, 7, 1]
    threshold_target=0.5,     # 层1：与最佳恶性簇相似度 >= 该值
    threshold_ratio=3.0,      # 层2：最佳恶性簇相似度 >= 良性簇相似度 * ratio
    save_rejected=False,      # 是否保存被拒绝的细胞（调试用）
    rejected_dir=None,
):
    """
    增强版细胞级筛选，在原 argmax 判断基础上增加两层规则：

    原逻辑（保留）：
        argmax(cell_sims) 必须在 target_ref_indices 里

    新增层1（门槛层）：
        与最佳恶性参考簇的相似度 >= threshold_target

    新增层2（专一层）：
        最佳恶性簇相似度 >= 所有良性参考簇相似度 * threshold_ratio
        （良性簇 = 不在 target_ref_indices 里的所有参考簇）
        注意：不惩罚"同时像多个恶性簇"的细胞
    """
    count = 0
    processed_indices = []

    # 统计
    stats = {
        'total'           : len(indices),
        'passed'          : 0,
        'rejected_argmax' : 0,   # 原逻辑：argmax 不在恶性簇里
        'rejected_layer1' : 0,   # 新增层1：恶性相似度不足
        'rejected_layer2' : 0,   # 新增层2：与良性簇太像
        'rejection_details': [],
    }

    n_ref = cell_sim_matrix.shape[1]
    benign_indices = [i for i in range(n_ref) if i not in target_ref_indices]

    for idx in indices:
        cell_sim_vec = cell_sim_matrix[idx]

        # ── 原逻辑：argmax 必须在恶性簇里 ──────────────────────────
        most_similar_ref = int(np.argmax(cell_sim_vec))
        if most_similar_ref not in target_ref_indices:
            stats['rejected_argmax'] += 1
            stats['rejection_details'].append({
                'idx'   : idx,
                'path'  : image_paths[idx],
                'reason': f'argmax=R{most_similar_ref} 不在恶性簇 {target_ref_indices} 里',
                'layer' : 0,
            })
            if save_rejected and rejected_dir:
                _copy_rejected(image_paths[idx], rejected_dir, "L0", idx)
            continue

        # ── 找出最佳恶性簇相似度 ────────────────────────────────────
        mal_sims     = cell_sim_vec[target_ref_indices]
        best_mal_sim = float(mal_sims.max())
        best_mal_idx = target_ref_indices[int(mal_sims.argmax())]

        # ── 新增层1：门槛检查 ────────────────────────────────────────
        if best_mal_sim < threshold_target:
            stats['rejected_layer1'] += 1
            stats['rejection_details'].append({
                'idx'   : idx,
                'path'  : image_paths[idx],
                'reason': f'层1: 恶性相似度 {best_mal_sim:.3f} < {threshold_target}',
                'layer' : 1,
            })
            if save_rejected and rejected_dir:
                _copy_rejected(image_paths[idx], rejected_dir, "L1", idx)
            continue

        # ── 新增层2：专一性检查（只和良性簇比） ──────────────────────
        if benign_indices:
            ben_sims    = cell_sim_vec[benign_indices]
            max_ben_sim = float(ben_sims.max())
            limit       = best_mal_sim / threshold_ratio

            if max_ben_sim >= limit:
                stats['rejected_layer2'] += 1
                stats['rejection_details'].append({
                    'idx'   : idx,
                    'path'  : image_paths[idx],
                    'reason': (f'层2: 良性最大相似度 {max_ben_sim:.3f} '
                               f'>= limit {limit:.3f} '
                               f'(恶性{best_mal_sim:.3f} / ratio{threshold_ratio})'),
                    'layer' : 2,
                })
                if save_rejected and rejected_dir:
                    _copy_rejected(image_paths[idx], rejected_dir, "L2", idx)
                continue

        # ── 全部通过，保存细胞 ───────────────────────────────────────
        src_path      = image_paths[idx]
        filename      = os.path.basename(src_path)
        cluster_label = labels[idx]
        prefix        = f"{rule_type}_cluster{cluster_label}_ref{best_mal_idx}"
        new_name      = f"{prefix}_{filename}"

        shutil.copy(src_path, os.path.join(save_dirs['rule_dir'],  new_name))
        shutil.copy(src_path, os.path.join(save_dirs['total_dir'], new_name))

        count += 1
        processed_indices.append(idx)
        stats['passed'] += 1

    # ── 打印统计 ─────────────────────────────────────────────────────
    total = stats['total']
    if total > 0:
        print(f"\n  [{rule_type}] 细胞级筛选统计（输入 {total} 个细胞）：")
        print(f"    ✅ 通过保留       : {stats['passed']:4d}  ({stats['passed']/total*100:5.1f}%)")
        print(f"    ❌ argmax非恶性簇 : {stats['rejected_argmax']:4d}  ({stats['rejected_argmax']/total*100:5.1f}%)")
        print(f"    ❌ 层1 恶性度不足 : {stats['rejected_layer1']:4d}  ({stats['rejected_layer1']/total*100:5.1f}%)")
        print(f"    ❌ 层2 良性簇太像 : {stats['rejected_layer2']:4d}  ({stats['rejected_layer2']/total*100:5.1f}%)")

    return count, processed_indices, stats


def _copy_rejected(src_path, rejected_dir, tag, idx):
    """辅助函数：把被拒绝的细胞复制到调试目录"""
    os.makedirs(rejected_dir, exist_ok=True)
    filename = os.path.basename(src_path)
    shutil.copy(src_path, os.path.join(rejected_dir, f"rejected_{tag}_{idx}_{filename}"))


print("✅ Block 4 save_matched_cells_v2 定义完成")
print(f"   层1 threshold_target : {CELL_THRESHOLD_TARGET}")
print(f"   层2 threshold_ratio  : {CELL_THRESHOLD_RATIO}")


# %%
# ============================================================
# Block 5 ：处理候选域（特征提取 + 聚类）
# ============================================================
pid      = extract_patient_id(CANDIDATE_FOLDER)
save_dir = os.path.join(BASE_SAVE_DIR, pid)
os.makedirs(save_dir, exist_ok=True)

print(f"======= 处理候选域: {pid} ({PATIENT_STATUS}) =======")

# 收集图像路径
cand_paths = collect_image_paths(os.path.join(CANDIDATE_FOLDER, "malignant_images"))
assert cand_paths, f"❌ 未找到细胞图像，请检查路径: {CANDIDATE_FOLDER}/malignant_images"

cand_sources = [0] * len(cand_paths)
cand_sizes   = np.log1p([
    count_non_background_pixels(p) for p in cand_paths
])

# 特征提取
cand_loader = DataLoader(
    CellImageDataset(cand_paths, transform=transform, source_labels=cand_sources),
    batch_size=BATCH_SIZE, shuffle=False
)
cand_feats, _, cand_paths = extract_features_with_cell_size(
    model, cand_loader, device, cand_sizes, FEATURE_LAYER, CELL_SIZE_WEIGHT
)

# PCA Transform（用参考域的 pca，不重新 fit）
cand_norm = normalize(pca.transform(cand_feats))

print(f"✅ Block 5 候选域特征提取完成")
print(f"   细胞总数: {len(cand_paths)}")
print(f"   特征维度: {cand_norm.shape}")
print(f"   {'⚠️  细胞数 < 32，将使用单细胞模式' if len(cand_norm) < 32 else '将使用聚类模式'}")


# %%
# ============================================================
# Block 6 ：簇级匹配（identify_matching_clusters）
#           + 相似度热力图可视化
# ============================================================

# ── 动态初始化保存目录 ──────────────────────────────────────
n_rules = len(MATCHING_RULES)
dirs = {f'rule{i+1}': os.path.join(save_dir, f'rule{i+1}_matched_cells') for i in range(n_rules)}
dirs['total'] = os.path.join(save_dir, 'matched_malignant_cells')
for d in dirs.values():
    os.makedirs(d, exist_ok=True)

if len(cand_norm) >= 32:
    # ── 聚类模式 ────────────────────────────────────────────
    cand_results, best_k = optimize_k_for_candidate_clustering(
        cand_norm, cand_sources, cand_paths, ref_results,
        save_dir, pid, MAX_CANDIDATE_K, MALIGNANT_REF_CLUSTERS
    )
    assert cand_results, "❌ 聚类优化失败，cand_results 为空"

    # 保存聚类明细
    pd.DataFrame({
        'labels'     : cand_results['labels'],
        'image_paths': cand_results['image_paths'],
    }).to_csv(os.path.join(save_dir, 'cluster_detail.csv'), index=False, encoding='utf-8-sig')

    # 簇级相似度矩阵
    *rule_clusters_list, similarity_matrix = identify_matching_clusters(
        cand_results['cluster_centers'],
        ref_results['cluster_centers'],
        MATCHING_RULES
    )

    # 细胞级相似度矩阵（用于后续细胞筛选）
    cell_sims = cosine_similarity(cand_results['features'], ref_results['cluster_centers'])

    # ── 打印簇级匹配结果 ────────────────────────────────────
    print(f"\n簇级匹配结果（共 {cand_results['n_clusters']} 个候选簇）：")
    for i, rule_clusters in enumerate(rule_clusters_list):
        rule = MATCHING_RULES[i]
        print(f"  Rule{i+1} (target=R{rule['target']}): 匹配到候选簇 {rule_clusters}")
        for c in rule_clusters:
            sim_val = similarity_matrix[c, rule['target']]
            print(f"    候选簇 C{c} → 与 R{rule['target']} 相似度 = {sim_val:.4f}")

    all_matched = [c for clusters in rule_clusters_list for c in clusters]
    unmatched   = [c for c in range(cand_results['n_clusters']) if c not in all_matched]
    print(f"  未匹配候选簇: {unmatched}")

    # ── 相似度热力图 ────────────────────────────────────────
    xlabel = [f"C{i}" for i in range(cand_results['n_clusters'])]
    ylabel = [f"R{i}" for i in range(ref_results['n_clusters'])]
    plot_similarity_heatmap(
        similarity_matrix.T, xlabel, ylabel,
        f"{pid} Cluster Similarity Matrix",
        os.path.join(save_dir, 'similarity_heatmap.png')
    )

    print(f"\n✅ Block 6 簇级匹配完成")

else:
    # ── 单细胞模式 ──────────────────────────────────────────
    cell_sims = cosine_similarity(cand_norm, ref_results['cluster_centers'])
    cand_results = None
    rule_clusters_list = None

    xlabel = [f"Cell{i}" for i in range(len(cell_sims))]
    ylabel = [f"R{i}" for i in range(ref_results['n_clusters'])]
    plot_similarity_heatmap(
        cell_sims.T, xlabel, ylabel,
        f"{pid} Cell Similarity Matrix",
        os.path.join(save_dir, 'Cell_similarity_heatmap.png')
    )
    print("✅ Block 6 单细胞模式，跳过聚类，直接进入细胞筛选")


# %%
# ============================================================
# Block 7 ：细胞级筛选（save_matched_cells_v2）
# 【调参重点】修改 Block 1 里的 CELL_THRESHOLD_TARGET / CELL_THRESHOLD_RATIO
# ============================================================

matched_counts = [0] * n_rules
all_stats      = {}

# ── 是否保存被拒绝的细胞（调试时建议开启）──────────────────
SAVE_REJECTED = True
rejected_dir  = os.path.join(save_dir, 'rejected_cells_debug')

print(f"======= 细胞级筛选 =======")
print(f"参数: threshold_target={CELL_THRESHOLD_TARGET}, threshold_ratio={CELL_THRESHOLD_RATIO}")

if cand_results is not None:
    # ── 聚类模式 ────────────────────────────────────────────
    for i, rule_clusters in enumerate(rule_clusters_list):
        rule_name = f'rule{i+1}'
        indices   = [j for j, lbl in enumerate(cand_results['labels']) if lbl in rule_clusters]

        count, _, stats = save_matched_cells_v2(
            indices          = indices,
            image_paths      = cand_paths,
            labels           = cand_results['labels'],
            cell_sim_matrix  = cell_sims,
            rule_type        = rule_name,
            save_dirs        = {'rule_dir': dirs[rule_name], 'total_dir': dirs['total']},
            target_ref_indices = MALIGNANT_REF_CLUSTERS,
            threshold_target = CELL_THRESHOLD_TARGET,
            threshold_ratio  = CELL_THRESHOLD_RATIO,
            save_rejected    = SAVE_REJECTED,
            rejected_dir     = os.path.join(rejected_dir, rule_name),
        )
        matched_counts[i] = count
        all_stats[rule_name] = stats

else:
    # ── 单细胞模式 ──────────────────────────────────────────
    processed = set()
    for i, rule in enumerate(MATCHING_RULES):
        rule_name = f'rule{i+1}'
        indices   = [
            j for j, s in enumerate(cell_sims)
            if j not in processed and check_matching_rules(
                s,
                ref_idx_target          = rule["target"],
                ref_idx_avoid           = rule["avoid"],
                threshold_target        = rule["threshold_target"],
                threshold_avoid_others  = rule.get("threshold_avoid_others"),
                threshold_ratio         = rule.get("threshold_ratio"),
            )
        ]
        count, newly_processed, stats = save_matched_cells_v2(
            indices          = indices,
            image_paths      = cand_paths,
            labels           = [0] * len(cand_paths),
            cell_sim_matrix  = cell_sims,
            rule_type        = rule_name,
            save_dirs        = {'rule_dir': dirs[rule_name], 'total_dir': dirs['total']},
            target_ref_indices = MALIGNANT_REF_CLUSTERS,
            threshold_target = CELL_THRESHOLD_TARGET,
            threshold_ratio  = CELL_THRESHOLD_RATIO,
            save_rejected    = SAVE_REJECTED,
            rejected_dir     = os.path.join(rejected_dir, rule_name),
        )
        matched_counts[i] = count
        all_stats[rule_name] = stats
        processed.update(newly_processed)

# ── 汇总打印 ─────────────────────────────────────────────────
total_matched = sum(matched_counts)
print(f"\n{'='*50}")
print(f"  病人: {pid} ({PATIENT_STATUS})")
print(f"  最终保留细胞数: {total_matched}")
for i in range(n_rules):
    print(f"    Rule{i+1}: {matched_counts[i]} 个")
print(f"{'='*50}")

if SAVE_REJECTED:
    print(f"\n  🔍 被拒绝的细胞已保存至: {rejected_dir}")
    print(f"     可人工查看各层拒绝的细胞图片，辅助调参")


# %%
# ============================================================
# Block 8 ：可视化结果（与 pipeline.py 完全一致）
# ============================================================

if cand_results is not None:
    # 构建 matching_pairs
    matching_pairs = []
    for rule, rule_clusters in zip(MATCHING_RULES, rule_clusters_list):
        target_ref = rule["target"]
        for cand_idx in rule_clusters:
            matching_pairs.append(
                (target_ref, cand_idx, similarity_matrix[cand_idx, target_ref])
            )

    rule_matched_clusters = {
        f'rule{i+1}': clusters for i, clusters in enumerate(rule_clusters_list)
    }

    calculate_consensus_score(ref_results, cand_results, matching_pairs, save_dir)

    visualize_matching_clusters(
        ref_results, cand_results, matching_pairs,
        save_dir, reference_umap, rule_matched_clusters
    )
    visualize_all_clusters_without_images(
        cand_results['features'], cand_results['labels'],
        cand_results['source_labels'], pid, save_dir, reference_umap
    )
    improved_visualize_clusters_with_images(
        cand_results['features'], cand_results['labels'],
        cand_results['source_labels'], cand_paths, pid, save_dir, cand_sizes
    )
    visualize_filtered_clusters(
        ref_results, cand_results, matching_pairs, save_dir,
        rule_matched_clusters  = rule_matched_clusters,
        target_ref_indices     = MALIGNANT_REF_CLUSTERS
    )

print(f"✅ Block 8 可视化完成，结果保存至: {save_dir}")


# %%
# ============================================================
# Block 9 ：相似度分布分析（调参辅助）
# 查看通过/被拒绝细胞的相似度分布，帮助决定阈值
# ============================================================
import matplotlib.pyplot as plt

print("======= 相似度分布分析 =======")

for rule_name, stats in all_stats.items():
    if stats['total'] == 0:
        continue

    # 收集所有细胞的最佳恶性相似度
    rule_idx   = int(rule_name.replace('rule', '')) - 1
    rule_clusters = rule_clusters_list[rule_idx] if cand_results else None

    if cand_results:
        indices_all = [j for j, lbl in enumerate(cand_results['labels'])
                       if lbl in rule_clusters]
    else:
        indices_all = list(range(len(cell_sims)))

    if not indices_all:
        continue

    mal_best_sims = []
    ben_max_sims  = []
    n_ref         = cell_sims.shape[1]
    benign_idx    = [i for i in range(n_ref) if i not in MALIGNANT_REF_CLUSTERS]

    for idx in indices_all:
        sv = cell_sims[idx]
        mal_best_sims.append(float(sv[MALIGNANT_REF_CLUSTERS].max()))
        if benign_idx:
            ben_max_sims.append(float(sv[benign_idx].max()))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f"{pid} - {rule_name} 细胞相似度分布（共 {len(indices_all)} 个细胞）")

    # 左图：最佳恶性相似度分布
    axes[0].hist(mal_best_sims, bins=30, color='tomato', edgecolor='white', alpha=0.8)
    axes[0].axvline(CELL_THRESHOLD_TARGET, color='red', linestyle='--',
                    label=f'threshold_target={CELL_THRESHOLD_TARGET}')
    axes[0].set_title('最佳恶性簇相似度分布')
    axes[0].set_xlabel('Cosine Similarity')
    axes[0].set_ylabel('细胞数')
    axes[0].legend()

    # 右图：恶性相似度 vs 良性相似度散点图
    if ben_max_sims:
        colors = ['green' if m / CELL_THRESHOLD_RATIO > b else 'red'
                  for m, b in zip(mal_best_sims, ben_max_sims)]
        axes[1].scatter(mal_best_sims, ben_max_sims, c=colors, alpha=0.5, s=10)
        # 绘制 ratio 分界线
        x_line = np.linspace(0, 1, 100)
        axes[1].plot(x_line, x_line / CELL_THRESHOLD_RATIO, 'b--',
                     label=f'ratio={CELL_THRESHOLD_RATIO} 分界线')
        axes[1].set_title('恶性相似度 vs 良性最大相似度\n绿=通过层2, 红=被层2拒绝')
        axes[1].set_xlabel('最佳恶性簇相似度')
        axes[1].set_ylabel('良性簇最大相似度')
        axes[1].legend()

    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f'{rule_name}_cell_sim_distribution.png'), dpi=150)
    plt.show()
    print(f"  {rule_name} 分布图已保存")